# sebal-et-gee — Day 4: Sensible Heat Flux H (iterative) + Daily ET — **MVP FINALE**

**Today we close the energy balance:**

$$\lambda ET = R_n - G - H$$

Where H is solved iteratively via the hot/cold anchor linearization with Monin-Obukhov stability correction. Then instantaneous ET is scaled to daily ET via the Evaporative Fraction (EF) method.

**Final deliverable:** a daily ET map (mm/day) over Thrace and station-level ET values — the MVP for outreach to Kilic, Anderson, Huntington.

## 1. Setup + rebuild Days 2-3 state

In [1]:
!pip install -q geemap geopandas

import ee
import geemap
import geopandas as gpd
import pandas as pd

EE_PROJECT = 'et-thrace'  # <-- REPLACE
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
print('Earth Engine ready.')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 6.3 MB/s eta 0:00:00
Earth Engine ready.


In [2]:
from google.colab import files
print('Upload thrace_boundary.geojson:')
uploaded = files.upload()

aoi_path = [f for f in uploaded if f.endswith(('.geojson', '.json'))][0]
aoi_gdf = gpd.read_file(aoi_path)
aoi_ee = geemap.geopandas_to_ee(aoi_gdf)
aoi_geom = aoi_ee.geometry()

stations = pd.DataFrame([
    {'name': 'Edirne',      'lat': 41.6667, 'lon': 26.5667, 'elevation_m':  51},
    {'name': 'Kirklareli',  'lat': 41.7333, 'lon': 27.2167, 'elevation_m': 232},
    {'name': 'Luleburgaz',  'lat': 41.4000, 'lon': 27.3500, 'elevation_m':  46},
    {'name': 'Uzunkopru',   'lat': 41.2667, 'lon': 26.6833, 'elevation_m':  22},
    {'name': 'Ipsala',      'lat': 40.9167, 'lon': 26.3833, 'elevation_m':  10},
    {'name': 'Corlu',       'lat': 41.1500, 'lon': 27.8000, 'elevation_m': 183},
    {'name': 'Tekirdag',    'lat': 40.9833, 'lon': 27.5167, 'elevation_m':   3},
    {'name': 'Sariyer',     'lat': 41.1667, 'lon': 29.0500, 'elevation_m':  59},
])
stations_ee = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([r['lon'], r['lat']]),
               {'name': r['name'], 'elevation_m': int(r['elevation_m'])})
    for _, r in stations.iterrows()
])
print('AOI + stations loaded.')


Upload thrace_boundary.geojson:


Saving thrace_stations.csv to thrace_stations.csv
Saving thrace_boundary.geojson to thrace_boundary.geojson
AOI + stations loaded.


In [3]:
# Rebuild Days 2-3 state in one block
def apply_scale_factors(image):
    optical = image.select('SR_B.').multiply(0.0000275).add(-0.2)
    thermal_k = image.select('ST_B10').multiply(0.00341802).add(149.0).rename('LST_K')
    thermal_c = thermal_k.subtract(273.15).rename('LST_C')
    return image.addBands(optical, None, True).addBands(thermal_k).addBands(thermal_c)

def mask_clouds(image):
    qa = image.select('QA_PIXEL')
    cloud = qa.bitwiseAnd(1 << 3).neq(0)
    shadow = qa.bitwiseAnd(1 << 4).neq(0)
    return image.updateMask(cloud.Or(shadow).Not())

COMMON_BANDS = ['SR_B2','SR_B3','SR_B4','SR_B5','SR_B6','SR_B7','ST_B10','QA_PIXEL']
ANCHOR = '2023-07-20'
WINDOW_DAYS = 14

anchor = ee.Date(ANCHOR)
start = anchor.advance(-WINDOW_DAYS // 2, 'day')
end = anchor.advance(WINDOW_DAYS // 2, 'day')

l8 = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').filterBounds(aoi_geom)
      .filterDate(start, end).filter(ee.Filter.lt('CLOUD_COVER', 80)).select(COMMON_BANDS))
l9 = (ee.ImageCollection('LANDSAT/LC09/C02/T1_L2').filterBounds(aoi_geom)
      .filterDate(start, end).filter(ee.Filter.lt('CLOUD_COVER', 80)).select(COMMON_BANDS))
merged = l8.merge(l9).map(mask_clouds).map(apply_scale_factors).sort('CLOUD_COVER')
mosaic = merged.mosaic().clip(aoi_geom)

# Albedo (Tasumi 2008)
def compute_albedo(img):
    coeffs = {'SR_B2': 0.300, 'SR_B3': 0.277, 'SR_B4': 0.233,
              'SR_B5': 0.143, 'SR_B6': 0.036, 'SR_B7': 0.012}
    terms = [img.select(b).multiply(c) for b, c in coeffs.items()]
    albedo = terms[0]
    for t in terms[1:]:
        albedo = albedo.add(t)
    return albedo.rename('albedo').clamp(0.0, 1.0)
albedo = compute_albedo(mosaic)

# NDVI + emissivity + LST
ndvi = mosaic.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI')
emissivity = ndvi.multiply(0.01).add(0.985).clamp(0.96, 0.99).rename('emissivity')
LST_K = mosaic.select('LST_K')

# ERA5-Land
era5_vars = ['temperature_2m', 'dewpoint_temperature_2m',
             'u_component_of_wind_10m', 'v_component_of_wind_10m',
             'surface_solar_radiation_downwards_hourly',
             'surface_thermal_radiation_downwards_hourly',
             'surface_pressure']

n_days = int(end.difference(start, 'day').getInfo())
daily_era5 = []
for i in range(n_days):
    d = start.advance(i, 'day')
    h0 = d.update(hour=10, minute=0, second=0)
    daily_era5.append(ee.ImageCollection('ECMWF/ERA5_LAND/HOURLY')
                      .filterDate(h0, h0.advance(1, 'hour'))
                      .select(era5_vars).first())
era5_mean = ee.ImageCollection(daily_era5).mean().clip(aoi_geom)

T_air_K = era5_mean.select('temperature_2m').rename('T_air_K')
u10 = era5_mean.select('u_component_of_wind_10m')
v10 = era5_mean.select('v_component_of_wind_10m')
wind_10m = u10.pow(2).add(v10.pow(2)).sqrt().rename('wind_10m')
Rs_in = era5_mean.select('surface_solar_radiation_downwards_hourly').divide(3600).rename('Rs_in')
Rl_in = era5_mean.select('surface_thermal_radiation_downwards_hourly').divide(3600).rename('Rl_in')
P_surf = era5_mean.select('surface_pressure').rename('P_surf')

SIGMA = 5.670374419e-8
Rl_out = emissivity.multiply(SIGMA).multiply(LST_K.pow(4)).rename('Rl_out')
Rn = (Rs_in.multiply(ee.Image(1).subtract(albedo))
      .add(Rl_in).subtract(Rl_out).rename('Rn'))

# Soil heat flux G
T_C = LST_K.subtract(273.15)
g_ratio = (T_C.divide(albedo)
           .multiply(albedo.multiply(0.0038).add(albedo.pow(2).multiply(0.0074)))
           .multiply(ndvi.pow(4).multiply(0.98).multiply(-1).add(1))).rename('g_ratio')
water = ndvi.lt(0)
g_ratio = g_ratio.where(water, 0.5).clamp(0.05, 0.5)
G = Rn.multiply(g_ratio).rename('G')
Rn_minus_G = Rn.subtract(G).rename('Rn_minus_G')

print('Day 2-3 state rebuilt. Rn, G, LST, albedo, meteorology all ready.')


Day 2-3 state rebuilt. Rn, G, LST, albedo, meteorology all ready.


In [4]:
# Recompute hot/cold anchors (Day 3 logic)
ag_mask = ndvi.gt(0.05).And(albedo.lt(0.30)).And(albedo.gt(0.10))
ag_stack = ee.Image.cat([ndvi, LST_K.rename('LST_K'), albedo, Rn, G]).updateMask(ag_mask)

pcts = ag_stack.reduceRegion(
    reducer=ee.Reducer.percentile([5, 20, 80, 95]),
    geometry=aoi_geom, scale=90, maxPixels=1e9, bestEffort=True
).getInfo()

cold_mask = (ag_stack.select('NDVI').gt(pcts['NDVI_p95'])
             .And(ag_stack.select('LST_K').lt(pcts['LST_K_p20'])))
hot_mask = (ag_stack.select('NDVI').gt(pcts['NDVI_p5'])
            .And(ag_stack.select('NDVI').lt(pcts['NDVI_p20']))
            .And(ag_stack.select('LST_K').gt(pcts['LST_K_p95'])))

cold_stats = ag_stack.updateMask(cold_mask).reduceRegion(
    reducer=ee.Reducer.median(), geometry=aoi_geom, scale=90,
    maxPixels=1e9, bestEffort=True).getInfo()
hot_stats = ag_stack.updateMask(hot_mask).reduceRegion(
    reducer=ee.Reducer.median(), geometry=aoi_geom, scale=90,
    maxPixels=1e9, bestEffort=True).getInfo()

print('Cold anchor:', {k: round(v, 3) for k, v in cold_stats.items() if v is not None})
print('Hot anchor: ', {k: round(v, 3) for k, v in hot_stats.items() if v is not None})


Cold anchor: {'G': 49.376, 'LST_K': 304.813, 'NDVI': 0.852, 'Rn': 694.505, 'albedo': 0.104}
Hot anchor:  {'G': 135.219, 'LST_K': 324.687, 'NDVI': 0.241, 'Rn': 523.495, 'albedo': 0.163}


## 2. Aerodynamic resistance $r_{ah}$ — neutral first pass

$$r_{ah} = \frac{\ln(z_2/z_1)}{u^* \cdot k}$$

with friction velocity $u^* = k \cdot u_{200} / \ln(200/z_{0m})$, where $z_{0m}$ is momentum roughness length (parameterized from NDVI), $z_1 = 0.1$ m, $z_2 = 2.0$ m, $k = 0.41$ (von Karman).

Wind at 200 m (blending height) is extrapolated from 10 m ERA5 wind under neutral conditions. First pass uses neutral stability; correction in the iteration loop.

In [5]:
K_VK = 0.41         # von Karman
Z1, Z2 = 0.1, 2.0   # heights for dT slab
Z_BLEND = 200.0     # blending height
Z_WIND = 10.0       # ERA5 wind height

# Momentum roughness length z0m from NDVI (Bastiaanssen parameterization)
# log10(z0m) = a + b * NDVI/albedo; here simpler: z0m = exp(a + b*NDVI)
# Use the Tasumi empirical: z0m = 0.018 * LAI, with LAI ~ NDVI-based SAVI approach simplified
# For MVP we use: z0m = 0.12 * h_crop, h_crop ~ 0.5 * NDVI m → z0m ~ 0.06 * NDVI
z0m = ndvi.multiply(0.06).max(ee.Image(0.005)).rename('z0m')  # floor 5 mm

# Extrapolate 10 m wind to blending height (200 m) under neutral conditions
# u_200 = u_10 * ln(200/z0m) / ln(10/z0m)
u_star_10 = wind_10m.multiply(K_VK).divide(z0m.multiply(Z_WIND).log().subtract(z0m.log()))
u_200 = u_star_10.divide(K_VK).multiply(z0m.multiply(Z_BLEND).log().subtract(z0m.log()))

# Friction velocity at the surface
u_star = u_200.multiply(K_VK).divide(ee.Image(Z_BLEND).divide(z0m).log()).rename('u_star')

# Neutral aerodynamic resistance r_ah between Z1 and Z2
r_ah_neutral = ee.Image(Z2/Z1).log().divide(u_star.multiply(K_VK)).rename('r_ah')

# Sanity check at stations
stn_ra = ee.Image.cat([z0m, u_star, r_ah_neutral]).reduceRegions(
    collection=stations_ee, reducer=ee.Reducer.mean(), scale=30)
df_ra = geemap.ee_to_df(stn_ra)[['name', 'z0m', 'u_star', 'r_ah']].round(3)
print('Station aerodynamic parameters (neutral):')
df_ra


Station aerodynamic parameters (neutral):


,name,z0m,u_star,r_ah
0,Edirne,0.007,0.084,86.614
1,Kirklareli,0.024,0.124,58.846
2,Luleburgaz,0.016,0.191,38.346
3,Uzunkopru,0.014,0.130,56.094
4,Ipsala,0.044,0.154,47.588
5,Corlu,0.015,0.199,36.633
6,Tekirdag,0.018,0.210,34.798
7,Sariyer,0.045,0.306,23.864


**Expected ranges:**
- `z0m`: 0.005–0.06 m (higher for dense vegetation)
- `u_star`: 0.2–0.8 m/s
- `r_ah`: 5–50 s/m (lower = more turbulent)

## 3. The dT linear relationship from anchor pixels

SEBAL's key insight: $dT = T_{z1} - T_{z2}$ is linearly related to LST across the image.

$$dT = a + b \cdot T_s$$

At the cold pixel, $\lambda ET = R_n - G$, so $H = 0$, so $dT_{cold} = 0$.
At the hot pixel, $\lambda ET \approx 0$, so $H = R_n - G$, so $dT_{hot} = H_{hot} \cdot r_{ah,hot} / (\rho \cdot c_p)$.

With two (T_s, dT) points, solve for a and b.

In [6]:
CP = 1004.0  # J/(kg·K) specific heat of air

# Air density using surface pressure and air temperature (ideal gas)
R_SPEC = 287.05
rho_air = P_surf.divide(T_air_K.multiply(R_SPEC)).rename('rho_air')  # kg/m³

# Extract r_ah at cold and hot anchors — use AOI-mean as MVP proxy
# (a more rigorous SEBAL extracts r_ah at the exact anchor pixel locations;
#  for MVP we take the spatial mean over the anchor candidate pools)
r_ah_cold = ee.Number(r_ah_neutral.updateMask(cold_mask).reduceRegion(
    reducer=ee.Reducer.median(), geometry=aoi_geom, scale=90,
    maxPixels=1e9, bestEffort=True).get('r_ah'))

r_ah_hot = ee.Number(r_ah_neutral.updateMask(hot_mask).reduceRegion(
    reducer=ee.Reducer.median(), geometry=aoi_geom, scale=90,
    maxPixels=1e9, bestEffort=True).get('r_ah'))

rho_cold = ee.Number(rho_air.updateMask(cold_mask).reduceRegion(
    reducer=ee.Reducer.median(), geometry=aoi_geom, scale=90,
    maxPixels=1e9, bestEffort=True).get('rho_air'))

rho_hot = ee.Number(rho_air.updateMask(hot_mask).reduceRegion(
    reducer=ee.Reducer.median(), geometry=aoi_geom, scale=90,
    maxPixels=1e9, bestEffort=True).get('rho_air'))

print('r_ah_cold =', r_ah_cold.getInfo(), 's/m')
print('r_ah_hot  =', r_ah_hot.getInfo(), 's/m')

# Anchor values from Day 3
Ts_cold = ee.Number(cold_stats['LST_K'])
Ts_hot = ee.Number(hot_stats['LST_K'])
Rn_cold = ee.Number(cold_stats['Rn'])
G_cold = ee.Number(cold_stats['G'])
Rn_hot = ee.Number(hot_stats['Rn'])
G_hot = ee.Number(hot_stats['G'])

# H at anchors
H_cold = ee.Number(0.0)  # by definition
H_hot = Rn_hot.subtract(G_hot)  # λET ≈ 0 at hot pixel

# dT at anchors: dT = H * r_ah / (rho * cp)
dT_cold = ee.Number(0.0)
dT_hot = H_hot.multiply(r_ah_hot).divide(rho_hot.multiply(CP))

# Linear fit: dT = a + b * Ts
b_slope = dT_hot.subtract(dT_cold).divide(Ts_hot.subtract(Ts_cold))
a_intercept = dT_cold.subtract(b_slope.multiply(Ts_cold))

print(f'\ndT linear fit: dT = {a_intercept.getInfo():.3f} + {b_slope.getInfo():.3f} * Ts')
print(f'  Ts_cold = {Ts_cold.getInfo():.2f} K → dT = 0 (by anchor def)')
print(f'  Ts_hot  = {Ts_hot.getInfo():.2f} K → dT = {dT_hot.getInfo():.3f} K')


r_ah_cold = 25.769140271768798 s/m
r_ah_hot  = 38.23546127620682 s/m

dT linear fit: dT = -199.831 + 0.656 * Ts
  Ts_cold = 304.81 K → dT = 0 (by anchor def)
  Ts_hot  = 324.69 K → dT = 13.030 K


**Sanity expectations:**
- `b_slope` (K/K): typically 0.05–0.30 for Mediterranean summer
- `dT_hot`: typically 3–10 K
- If `dT_hot` is negative or > 20 K, anchor selection likely failed — investigate before proceeding.

## 4. Iterative H solution with Monin-Obukhov stability

Initial H is computed from the neutral $r_{ah}$. Then we correct $r_{ah}$ for atmospheric stability using Monin-Obukhov similarity theory and recompute H. Iterate until convergence (typically 3-5 iterations).

Stability parameter: $L = -\rho c_p u^{*3} T_s / (k g H)$ (Monin-Obukhov length)

Stability correction functions $\psi_m$ and $\psi_h$ for unstable conditions (daytime when L < 0):

$$x = (1 - 16z/L)^{1/4}$$
$$\psi_m(z) = 2\ln\frac{1+x}{2} + \ln\frac{1+x^2}{2} - 2\arctan(x) + \pi/2$$
$$\psi_h(z) = 2\ln\frac{1+x^2}{2}$$

Then corrected $r_{ah}$ and $u^*$ are computed. **This is the classic SEBAL/METRIC stability iteration.**

In [11]:
# --- CIMEC H iteration (replaces the previous H iteration block) ---
G_GRAV = 9.81
N_ITER = 8

def psi_m_unstable(z, L):
    x = ee.Image(1).subtract(L.pow(-1).multiply(z).multiply(16)).pow(0.25)
    return (x.add(1).divide(2).log().multiply(2)
            .add(x.pow(2).add(1).divide(2).log())
            .subtract(x.atan().multiply(2))
            .add(ee.Image(3.14159265 / 2)))

def psi_h_unstable(z, L):
    x = ee.Image(1).subtract(L.pow(-1).multiply(z).multiply(16)).pow(0.25)
    return x.pow(2).add(1).divide(2).log().multiply(2)

def get_anchor_value(img, mask):
    """Extract median value of a single-band image at the anchor pixel pool."""
    return ee.Number(img.updateMask(mask).reduceRegion(
        reducer=ee.Reducer.median(), geometry=aoi_geom, scale=90,
        maxPixels=1e9, bestEffort=True).values().get(0))

# Fixed energy boundary conditions at anchors
Rn_hot_val = ee.Number(hot_stats['Rn'])
G_hot_val = ee.Number(hot_stats['G'])
H_hot_fixed = Rn_hot_val.subtract(G_hot_val)   # λET ≈ 0 at hot
H_cold_fixed = ee.Number(0.0)                  # λET = Rn-G at cold

Ts_cold = ee.Number(cold_stats['LST_K'])
Ts_hot = ee.Number(hot_stats['LST_K'])

# Initial: neutral r_ah everywhere
r_ah = r_ah_neutral
u_star_iter = u_star

print('Starting CIMEC iterative H solution...')
print(f'H_hot_fixed (boundary) = {H_hot_fixed.getInfo():.1f} W/m²')

for i in range(N_ITER):
    # Anchor r_ah and rho with current r_ah/u_star
    r_ah_cold_i = get_anchor_value(r_ah, cold_mask)
    r_ah_hot_i = get_anchor_value(r_ah, hot_mask)
    rho_cold_i = get_anchor_value(rho_air, cold_mask)
    rho_hot_i = get_anchor_value(rho_air, hot_mask)

    # dT at anchors derived from FIXED H boundary conditions
    dT_cold_i = ee.Number(0.0)
    dT_hot_i = H_hot_fixed.multiply(r_ah_hot_i).divide(rho_hot_i.multiply(CP))

    # Linear fit dT = a + b * Ts
    b_slope = dT_hot_i.subtract(dT_cold_i).divide(Ts_hot.subtract(Ts_cold))
    a_intercept = dT_cold_i.subtract(b_slope.multiply(Ts_cold))

    # dT everywhere
    dT = LST_K.multiply(b_slope).add(a_intercept).rename('dT')

    # H everywhere with CURRENT r_ah
    H = rho_air.multiply(CP).multiply(dT).divide(r_ah).rename('H')
    H = H.max(ee.Image(1.0))

    # Monin-Obukhov length L
    L = (rho_air.multiply(CP).multiply(LST_K).multiply(u_star_iter.pow(3))
         .divide(H.multiply(K_VK).multiply(G_GRAV)).multiply(-1)).rename('L')

    unstable = L.lt(0)
    psi_m_200 = psi_m_unstable(Z_BLEND, L).updateMask(unstable).unmask(0)
    psi_h_2 = psi_h_unstable(Z2, L).updateMask(unstable).unmask(0)
    psi_h_01 = psi_h_unstable(Z1, L).updateMask(unstable).unmask(0)

    # Corrected u* and r_ah
    u_star_iter = (u_200.multiply(K_VK)
                   .divide(ee.Image(Z_BLEND).divide(z0m).log().subtract(psi_m_200)))
    r_ah = (ee.Image(Z2/Z1).log().subtract(psi_h_2).add(psi_h_01)
            .divide(u_star_iter.multiply(K_VK))).rename('r_ah')
    r_ah = r_ah.max(ee.Image(1.0))

    print(f'  Iter {i+1}/{N_ITER}: '
          f'b_slope={b_slope.getInfo():.4f}, '
          f'dT_hot={dT_hot_i.getInfo():.2f} K, '
          f'r_ah_hot={r_ah_hot_i.getInfo():.1f} s/m')

# Final H
H = rho_air.multiply(CP).multiply(dT).divide(r_ah).rename('H')
H = H.max(ee.Image(0)).min(Rn_minus_G)

# Final linear fit parameters (for reporting)
final_a = a_intercept.getInfo()
final_b = b_slope.getInfo()
print(f'\nFINAL dT fit: dT = {final_a:.3f} + {final_b:.4f} * Ts')

stn_H = H.reduceRegions(collection=stations_ee, reducer=ee.Reducer.mean(), scale=30)
df_H = geemap.ee_to_df(stn_H)[['name', 'mean']].rename(columns={'mean': 'H_Wm2'}).round(1)
print('\nStation-level sensible heat flux H (W/m²):')
df_H

Starting CIMEC iterative H solution...
H_hot_fixed (boundary) = 388.3 W/m²
  Iter 1/8: b_slope=0.6556, dT_hot=13.03 K, r_ah_hot=38.2 s/m
  Iter 2/8: b_slope=0.1520, dT_hot=3.02 K, r_ah_hot=8.9 s/m
  Iter 3/8: b_slope=0.3434, dT_hot=6.83 K, r_ah_hot=20.0 s/m
  Iter 4/8: b_slope=0.2585, dT_hot=5.14 K, r_ah_hot=15.1 s/m
  Iter 5/8: b_slope=0.2907, dT_hot=5.78 K, r_ah_hot=17.0 s/m
  Iter 6/8: b_slope=0.2778, dT_hot=5.52 K, r_ah_hot=16.2 s/m
  Iter 7/8: b_slope=0.2826, dT_hot=5.62 K, r_ah_hot=16.5 s/m
  Iter 8/8: b_slope=0.2810, dT_hot=5.59 K, r_ah_hot=16.4 s/m

FINAL dT fit: dT = -85.662 + 0.2810 * Ts

Station-level sensible heat flux H (W/m²):


,name,H_Wm2
0,Edirne,303.2
1,Kirklareli,139.7
2,Luleburgaz,230.7
3,Uzunkopru,235.5
4,Ipsala,108.8
5,Corlu,264.3
6,Tekirdag,235.0
7,Sariyer,99.5


## 5. Latent heat flux λET and instantaneous ET

In [13]:
# Latent heat of vaporization (temperature-dependent)
LAMBDA = T_air_K.subtract(273.15).multiply(-0.00236).add(2.501).multiply(1e6)  # J/kg

# λET = Rn - G - H
LE = Rn_minus_G.subtract(H).rename('LE')
LE = LE.max(ee.Image(0))  # physical floor

# Instantaneous ET in mm/hour: ET_inst = 3600 * LE / (lambda * 1000)
# (1000 = water density kg/m³)
ET_inst = LE.divide(LAMBDA).multiply(3600).rename('ET_inst_mm_hr')

stn_LE = LE.reduceRegions(collection=stations_ee, reducer=ee.Reducer.mean(), scale=30)
df_LE = geemap.ee_to_df(stn_LE)[['name', 'mean']].rename(columns={'mean': 'LE_Wm2'}).round(1)
print('Station-level latent heat flux λET (W/m²):')
df_LE


Station-level latent heat flux λET (W/m²):


,name,LE_Wm2
0,Edirne,113.6
1,Kirklareli,347.7
2,Luleburgaz,243.6
3,Uzunkopru,221.2
4,Ipsala,488.0
5,Corlu,201.9
6,Tekirdag,252.9
7,Sariyer,509.7


## 6. Daily ET via Evaporative Fraction (EF)

The classic SEBAL assumption: evaporative fraction $EF = \lambda ET / (R_n - G)$ is nearly constant through the daytime.

So daily ET (mm/day) can be computed as:

$$ET_{daily} = EF \cdot \frac{86400 \cdot R_{n,daily}}{\lambda \cdot \rho_w}$$

Where $R_{n,daily}$ is the 24-hour mean net radiation from ERA5-Land (in W/m²), computed from the same 14-day window as a daily mean.

**Key simplification for MVP:** daily Rn ≈ 0.3 × instantaneous midday Rn (typical daytime-to-daily ratio for Mediterranean summer). A more rigorous version integrates ERA5 Rn over 24h; we use the proxy here for speed.

In [14]:
# Evaporative fraction
EF = LE.divide(Rn_minus_G).rename('EF')
EF = EF.clamp(0, 1)  # physical bounds

# Compute 24-hour mean ERA5-Land net radiation over the 14-day window
# ERA5-Land surface_net_solar_radiation_hourly and surface_net_thermal_radiation_hourly
era5_rn_vars = ['surface_net_solar_radiation_hourly',
                'surface_net_thermal_radiation_hourly']

era5_full = (ee.ImageCollection('ECMWF/ERA5_LAND/HOURLY')
             .filterDate(start, end)
             .select(era5_rn_vars))

# Mean hourly accumulation over window, converted from J/m²/h to W/m²
era5_rn_daily_mean = era5_full.mean().divide(3600).clip(aoi_geom)
Rn_24h = (era5_rn_daily_mean.select('surface_net_solar_radiation_hourly')
          .add(era5_rn_daily_mean.select('surface_net_thermal_radiation_hourly'))
          .rename('Rn_24h'))

# Daily ET in mm/day: ET = EF * 86400 * Rn_24h / (lambda * 1000)
ET_daily = (EF.multiply(Rn_24h).multiply(86400)
            .divide(LAMBDA).rename('ET_daily_mm'))
ET_daily = ET_daily.max(ee.Image(0))

stn_ETd = ET_daily.reduceRegions(collection=stations_ee, reducer=ee.Reducer.mean(), scale=30)
df_ETd = geemap.ee_to_df(stn_ETd)[['name', 'mean']].rename(columns={'mean': 'ET_mm_day'}).round(2)
print('Station-level DAILY ET (mm/day) — 2023-07-20 ± 7 days:')
df_ETd


Station-level DAILY ET (mm/day) — 2023-07-20 ± 7 days:


,name,ET_mm_day
0,Edirne,1.40
1,Kirklareli,3.91
2,Luleburgaz,2.74
3,Uzunkopru,2.58
4,Ipsala,4.42
5,Corlu,2.36
6,Tekirdag,2.88
7,Sariyer,5.62


**Expected daily ET for Thrace mid-summer:**
- Irrigated cropland (sunflower, corn): 4–7 mm/day
- Rainfed wheat stubble (post-harvest): 0.5–2 mm/day
- Forest/park: 3–5 mm/day
- Water bodies: 4–6 mm/day
- Urban: 1–3 mm/day

## 7. Final ET map — MVP deliverable

In [15]:
m = geemap.Map(center=[41.3, 27.5], zoom=8)
m.add_basemap('SATELLITE')

m.addLayer(mosaic, {'bands': ['SR_B4', 'SR_B3', 'SR_B2'], 'min': 0, 'max': 0.3}, 'Landsat RGB', False)
m.addLayer(ndvi, {'min': 0, 'max': 0.9, 'palette': ['brown', 'white', 'green']}, 'NDVI', False)
m.addLayer(Rn, {'min': 300, 'max': 800,
                'palette': ['000080', '0000ff', '00ffff', 'ffff00', 'ff8000', 'ff0000']},
           'Rn (W/m²)', False)
m.addLayer(H, {'min': 0, 'max': 500,
               'palette': ['white', 'yellow', 'orange', 'red', 'darkred']},
           'Sensible heat H (W/m²)', False)
m.addLayer(LE, {'min': 0, 'max': 600,
                'palette': ['white', 'lightblue', 'blue', 'darkblue', 'purple']},
           'Latent heat λET (W/m²)', False)
m.addLayer(EF, {'min': 0, 'max': 1,
                'palette': ['brown', 'yellow', 'lightgreen', 'green', 'darkgreen']},
           'Evaporative Fraction', False)

# THE MVP OUTPUT — daily ET in mm/day
et_palette = ['d73027', 'fc8d59', 'fee090', 'e0f3f8', '91bfdb', '4575b4', '313695']
m.addLayer(ET_daily, {'min': 0, 'max': 7, 'palette': et_palette},
           'DAILY ET (mm/day) — MVP OUTPUT')

m.addLayer(aoi_ee, {'color': 'yellow'}, 'Thrace AOI')
m.addLayer(stations_ee, {'color': 'red'}, '8 stations')

m.addLayerControl()
m


Map(center=[41.3, 27.5], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', …

## 8. Export ET map as GeoTIFF (for the repo + email attachment)

In [16]:
# Start a Drive export task — completes in a few minutes
task = ee.batch.Export.image.toDrive(
    image=ET_daily.toFloat(),
    description='Thrace_Daily_ET_2023-07-20',
    folder='sebal_et_gee_exports',
    region=aoi_geom,
    scale=30,
    maxPixels=1e10,
    fileFormat='GeoTIFF',
)
task.start()
print('Export task started. Check Google Drive > sebal_et_gee_exports folder in a few minutes.')
print(f'Task ID: {task.id}')
print('To check status: task.status()')


Export task started. Check Google Drive > sebal_et_gee_exports folder in a few minutes.
Task ID: 7KLF3PQZORU7UHQS3Y4Q5NCA
To check status: task.status()


## 9. MVP Summary Table — everything in one place

In [17]:
# Combine all station-level results
all_bands = ee.Image.cat([ndvi, albedo, LST_K.rename('LST_K'), Rn, G, H, LE, EF, ET_daily])
stn_all = all_bands.reduceRegions(collection=stations_ee, reducer=ee.Reducer.mean(), scale=30)
df_all = geemap.ee_to_df(stn_all)[
    ['name', 'NDVI', 'albedo', 'LST_K', 'Rn', 'G', 'H', 'LE', 'EF', 'ET_daily_mm']
]
df_all['LST_C'] = (df_all['LST_K'] - 273.15).round(1)
df_all = df_all[['name', 'NDVI', 'albedo', 'LST_C', 'Rn', 'G', 'H', 'LE', 'EF', 'ET_daily_mm']].round(2)
print('=' * 80)
print('SEBAL MVP — Thrace — 2023-07-20 ± 7 days — Full station summary')
print('=' * 80)
df_all


SEBAL MVP — Thrace — 2023-07-20 ± 7 days — Full station summary


,name,NDVI,albedo,LST_C,Rn,G,H,LE,EF,ET_daily_mm
0,Edirne,0.11,0.16,48.9,550.65,133.92,303.16,113.57,0.27,1.40
1,Kirklareli,0.41,0.14,41.8,605.75,118.41,139.65,347.69,0.71,3.91
2,Luleburgaz,0.27,0.13,45.1,602.86,128.56,230.69,243.61,0.51,2.74
3,Uzunkopru,0.23,0.14,46.3,589.41,132.67,235.53,221.20,0.48,2.58
4,Ipsala,0.74,0.09,39.5,682.22,85.35,108.84,488.04,0.82,4.42
5,Corlu,0.24,0.13,46.5,596.87,130.73,264.27,201.86,0.43,2.36
6,Tekirdag,0.31,0.12,44.8,615.07,127.18,235.00,252.88,0.52,2.88
7,Sariyer,0.75,0.09,36.9,687.41,78.22,99.46,509.73,0.84,5.62


## 10. Day 4 MVP deliverables checklist

- [ ] Aerodynamic resistance computed with roughness parameterization
- [ ] Hot/cold dT anchors define a physically valid linear fit
- [ ] Iterative H with Monin-Obukhov stability converged
- [ ] Station-level H: 50–400 W/m² (irrigated low, dry high)
- [ ] Station-level λET: 100–500 W/m² (irrigated high, dry low)
- [ ] Station-level daily ET: 1–7 mm/day range
- [ ] ET map exported to Drive as GeoTIFF

**🎉 If all checks pass, the MVP is complete.** You have an end-to-end, modular, Google Earth Engine-based METRIC/SEBAL-style ET pipeline over the Thrace region, aligned with the 8 stations in your Scientific Reports manuscript.

**Next (Day 5):** Refactor the notebook into a clean Python package (the 9-module structure we discussed), write the README, and push to GitHub as `sebal-et-gee`. That repo + the ET map screenshot + one demo notebook is the portfolio artifact you'll link in the email to Ayse Kilic.